# Crescent Bakery: Hypothesis Tests

Lesson 1.6 Guided Example. Runs t-tests on the Crescent Bakery dataset,
demonstrates the multiple testing problem with simulation, and shows how
Bonferroni correction works.

Author: Meron Welderufael
Date: 08/17/2026

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

In [3]:
bakery = pd.read_csv("C:/Users/Lenovo/northstar-coursework/module-01-foundations-of-analytics-and-statistics/lesson-1-2-types-of-data/bakery_customers.csv")

downtown = bakery[bakery['region'] == 'Downtown']['total_spent_usd']
west_end = bakery[bakery['region'] == 'West End']['total_spent_usd']

print(f"Downtown: n={len(downtown)}, mean=${downtown.mean():.2f}, std=${downtown.std():.2f}")
print(f"West End: n={len(west_end)}, mean=${west_end.mean():.2f}, std=${west_end.std():.2f}")


Downtown: n=24, mean=$253.00, std=$67.75
West End: n=6, mean=$253.13, std=$89.69


In [4]:
t_stat, p_value = stats.ttest_ind(downtown, west_end)

print(f"t-statistics: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")


t-statistics: -0.0040
p-value: 0.9969


In [5]:
mean_diff = downtown.mean() - west_end.mean()

#pooled standard error of the difference

n1, n2 = len(downtown), len(west_end)
s1, s2 = downtown.std(ddof=1), west_end.std(ddof=  1)
se_diff = np.sqrt(s1**2/n1 + s2**2/n2)

# 95% CI for the difference
margin = 1.96 * se_diff
ci_lower = mean_diff - margin
ci_upper = mean_diff + margin

print(f"Mean difference (Downtown - West End): ${mean_diff:.2f}")
print(f"95% CI for the difference: (${ci_lower:.2f}, ${ci_upper:.2f})")
print(f"p-value: {p_value:.4f}")


Mean difference (Downtown - West End): $-0.13
95% CI for the difference: ($-76.84, $76.58)
p-value: 0.9969


In [6]:
np.random.seed(2024)

n_tests = 20
significant_count = 0
all_p_values = []

for i in range(n_tests):
    # Two samples from the SAME distribution (null is true)
    a = np.random.normal(loc=100, scale=15, size=80)
    b = np.random.normal(loc=100, scale=15, size=80)

    _, p = stats.ttest_ind(a, b)
    all_p_values.append(p)

    if p < 0.05:
        significant_count += 1
        print(f"Test {i+1}: p = {p:.4f} (SIGNIFICANT, but null is actually true)")

print(f"\nOf {n_tests} tests on identical distributions, {significant_count} came back 'significant' at p < 0.05")

Test 7: p = 0.0193 (SIGNIFICANT, but null is actually true)

Of 20 tests on identical distributions, 1 came back 'significant' at p < 0.05


In [7]:
bonferroni_threshold = 0.05 / n_tests
significant_after_bonferroni = sum(p < bonferroni_threshold for p in all_p_values)

print(f"Original threshold: 0.05")
print(f"Bonferroni-corrected threshold: {bonferroni_threshold:.4f}")
print(f"Number significant at original threshold: {significant_count}")
print(f"Number significant after Bonferroni: {significant_after_bonferroni}")

Original threshold: 0.05
Bonferroni-corrected threshold: 0.0025
Number significant at original threshold: 1
Number significant after Bonferroni: 0


In [8]:
np.random.seed(0)

n_simulations = 500
n_tests_per_sim = 20

at_least_one_significant = 0

for _ in range(n_simulations):
    found_one = False
    for _ in range(n_tests_per_sim):
        a = np.random.normal(loc=100, scale=15, size=80)
        b = np.random.normal(loc=100, scale=15, size=80)
        _, p = stats.ttest_ind(a, b)
        if p < 0.05:
            found_one = True
            break
    if found_one:
        at_least_one_significant += 1

probability = at_least_one_significant / n_simulations
print(f"Of {n_simulations} simulations, each with {n_tests_per_sim} tests:")
print(f"At least one 'significant' result appeared in {at_least_one_significant} simulations")
print(f"Empirical probability of at least one false positive across 20 tests: {probability:.1%}")

Of 500 simulations, each with 20 tests:
At least one 'significant' result appeared in 330 simulations
Empirical probability of at least one false positive across 20 tests: 66.0%
